# Any-wasting prediction: CatBoost, XGBoost and LightGBM

This notebook compares three boosted-tree models using the same 23 predictors. It uses SMOTENC within training folds only, the MICS child sampling weight (ch_weight, with chweight fallback) during fitting and weighted evaluation, and a model-specific classification threshold selected by maximizing survey-weighted F1 on training-only out-of-fold predictions. Each model receives 30 Optuna trials.

**Weight caveat:** synthetic SMOTENC rows have no original survey weight. They are assigned the mean minority-class weight in that training fold; original rows retain their weights. This is an explicit approximation, not a survey-design estimator. Include a no-SMOTE sensitivity analysis before making substantive claims.

## 1. Install dependencies

In [ ]:
import importlib.util, subprocess, sys
packages = {"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn","catboost":"catboost","optuna":"optuna","imblearn":"imbalanced-learn","xgboost":"xgboost","lightgbm":"lightgbm","matplotlib":"matplotlib"}
missing = [pip for mod,pip in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Dependencies ready.")

## 2. Imports and settings

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, optuna
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix, classification_report,
    precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED, OPTUNA_TRIALS, CV_FOLDS = 42, 30, 5
np.random.seed(SEED)

## 3. Load dataset

In [ ]:
DATA_PATH = Path("MICS-child-ISRC-revised.dta")
if not DATA_PATH.exists():
    candidates = [p for p in [Path("MICS-child-ISRC.dta"), *Path(".").glob("*.dta")] if p.exists()]
    candidates = list(dict.fromkeys(candidates))
    if len(candidates) != 1:
        raise FileNotFoundError(f"Set DATA_PATH explicitly; found: {candidates}")
    DATA_PATH = candidates[0]
df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print(DATA_PATH, df.shape)
print("Weight-like fields:", [c for c in df.columns if "weight" in c.lower()])

## 4. Outcome, predictors and child sample weight

In [ ]:
TARGET = "any_wasting"
PREDICTORS = ["age_m","female","division","urban","wb4","parity_cat","diarrhoea_2w",
"cough_2w","breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level","electricity",
"wealth","birth_order","hh_size","water_treated","water_boil_filter","handwash_place",
"handwash_water","floor_quality","ari_chest"]
WEIGHT_CANDIDATES = ["ch_weight", "chweight"]
WEIGHT_COL = next((c for c in WEIGHT_CANDIDATES if c in df.columns), None)
if WEIGHT_COL is None:
    raise KeyError("Child weight not found. Expected ch_weight or chweight; verify the Stata variable name.")
required = [TARGET] + PREDICTORS + [WEIGHT_COL]
missing = [c for c in required if c not in df.columns]
if missing: raise KeyError(f"Missing columns: {missing}")
model_df = df[required].copy()
model_df[WEIGHT_COL] = pd.to_numeric(model_df[WEIGHT_COL], errors="coerce")
model_df = model_df.dropna(subset=required)
model_df = model_df.loc[model_df[WEIGHT_COL] > 0].copy()
model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="raise").astype(int)
if sorted(model_df[TARGET].unique()) != [0,1]: raise ValueError("Outcome must be coded 0/1.")
print("Weight field:", WEIGHT_COL, "| rows:", len(model_df))
print("Outcome counts:", model_df[TARGET].value_counts().sort_index().to_dict())
display(model_df[WEIGHT_COL].describe().to_frame("child sample weight"))

## 5. Feature types and stratified holdout split

In [ ]:
CATEGORICAL_COLUMNS = ["female","division","urban","parity_cat","diarrhoea_2w","cough_2w",
"breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level","electricity","wealth",
"birth_order","water_treated","water_boil_filter","handwash_place","handwash_water",
"floor_quality","ari_chest"]
NUMERIC_COLUMNS = [c for c in PREDICTORS if c not in CATEGORICAL_COLUMNS]
X = model_df[PREDICTORS].copy()
for c in CATEGORICAL_COLUMNS: X[c] = X[c].astype(str)
for c in NUMERIC_COLUMNS: X[c] = pd.to_numeric(X[c], errors="raise")
y = model_df[TARGET].astype(int)
w = model_df[WEIGHT_COL].astype(float)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.20, random_state=SEED, stratify=y)
print("Train:", len(X_train), "events:", int(y_train.sum()), "| test:", len(X_test), "events:", int(y_test.sum()))

## 6. SMOTENC and weighted model fitting

SMOTENC runs only on the current training fold. Original observations retain their child weights. Synthetic rows receive the mean minority-class weight from that fold. Weights are normalized to mean 1 for numerical stability.

In [ ]:
CAT_INDICES = [X.columns.get_loc(c) for c in CATEGORICAL_COLUMNS]
def smotenc_resample(Xf, yf, wf):
    counts = pd.Series(yf).value_counts()
    if counts.min() < 2: raise ValueError("Not enough minority observations for SMOTENC.")
    sampler = SMOTENC(categorical_features=CAT_INDICES, random_state=SEED,
                      k_neighbors=min(5, int(counts.min())-1))
    Xr, yr = sampler.fit_resample(Xf, yf)
    n_new = len(yr)-len(yf)
    minority = counts.idxmin()
    wr = np.r_[np.asarray(wf, dtype=float),
               np.repeat(np.asarray(wf, dtype=float)[np.asarray(yf)==minority].mean(), n_new)]
    return Xr, np.asarray(yr, dtype=int), wr/wr.mean()

def make_preprocessor():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
        ("num", "passthrough", NUMERIC_COLUMNS)
    ])

def build_model(name, p):
    if name == "CatBoost":
        return CatBoostClassifier(iterations=1200, loss_function="Logloss",
            eval_metric="PRAUC", random_seed=SEED, verbose=False,
            allow_writing_files=False, thread_count=1, **p)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=800, objective="binary:logistic",
            eval_metric="logloss", tree_method="hist", n_jobs=1,
            random_state=SEED, verbosity=0, **p)
    return LGBMClassifier(n_estimators=800, objective="binary",
        random_state=SEED, verbosity=-1, n_jobs=1, **p)

def fit_bundle(name, params, Xf, yf, wf):
    Xr, yr, wr = smotenc_resample(Xf, yf, wf)
    model = build_model(name, params)
    if name == "CatBoost":
        model.fit(Xr, yr, cat_features=CATEGORICAL_COLUMNS, sample_weight=wr)
        return {"model":model, "prep":None}
    prep = make_preprocessor()
    model.fit(prep.fit_transform(Xr), yr, sample_weight=wr)
    return {"model":model, "prep":prep}

def predict_bundle(bundle, name, Xe):
    Xe2 = Xe if name == "CatBoost" else bundle["prep"].transform(Xe)
    return bundle["model"].predict_proba(Xe2)[:,1]

## 7. Optuna tuning (30 trials per model; weighted PR-AUC objective)

In [ ]:
def suggest_params(t, name):
    if name == "CatBoost":
        return {"depth":t.suggest_int("depth",4,10),
          "learning_rate":t.suggest_float("learning_rate",.01,.20,log=True),
          "l2_leaf_reg":t.suggest_float("l2_leaf_reg",.1,30,log=True),
          "random_strength":t.suggest_float("random_strength",.001,10,log=True),
          "bagging_temperature":t.suggest_float("bagging_temperature",0,5),
          "border_count":t.suggest_categorical("border_count",[64,96,128,192,254])}
    if name == "XGBoost":
        return {"max_depth":t.suggest_int("max_depth",3,10),
          "learning_rate":t.suggest_float("learning_rate",.01,.20,log=True),
          "min_child_weight":t.suggest_float("min_child_weight",1,12),
          "subsample":t.suggest_float("subsample",.65,1),
          "colsample_bytree":t.suggest_float("colsample_bytree",.65,1),
          "reg_lambda":t.suggest_float("reg_lambda",.1,30,log=True)}
    return {"num_leaves":t.suggest_int("num_leaves",15,63),
      "max_depth":t.suggest_int("max_depth",-1,12),
      "learning_rate":t.suggest_float("learning_rate",.01,.20,log=True),
      "min_child_samples":t.suggest_int("min_child_samples",10,60),
      "subsample":t.suggest_float("subsample",.65,1),
      "colsample_bytree":t.suggest_float("colsample_bytree",.65,1),
      "reg_lambda":t.suggest_float("reg_lambda",.1,30,log=True)}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
def cv_score(name, params):
    scores = []
    for tr, va in cv.split(X_train, y_train):
        b = fit_bundle(name, params, X_train.iloc[tr], y_train.iloc[tr], w_train.iloc[tr])
        pr = predict_bundle(b, name, X_train.iloc[va])
        scores.append(average_precision_score(y_train.iloc[va], pr, sample_weight=w_train.iloc[va]))
    return float(np.mean(scores))

def tune(name):
    st = optuna.create_study(direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED), study_name=name+"_any_wasting")
    st.optimize(lambda t: cv_score(name, suggest_params(t,name)),
                n_trials=OPTUNA_TRIALS, n_jobs=1, show_progress_bar=True)
    return st

studies = {}
for name in ["CatBoost","XGBoost","LightGBM"]:
    print("\nTuning", name)
    studies[name] = tune(name)
    print("Best weighted CV PR-AUC:", studies[name].best_value)
    print(studies[name].best_params)

## 8. Optimize classification thresholds using training-only OOF predictions

For each tuned model, choose the threshold maximizing survey-weighted F1 across out-of-fold predictions from the training partition. The held-out test data are not used for threshold selection.

In [ ]:
def oof_predictions(name, params):
    oof = np.full(len(X_train), np.nan)
    for tr, va in cv.split(X_train, y_train):
        b = fit_bundle(name, params, X_train.iloc[tr], y_train.iloc[tr], w_train.iloc[tr])
        oof[va] = predict_bundle(b, name, X_train.iloc[va])
    if np.isnan(oof).any(): raise RuntimeError("Missing OOF predictions.")
    return oof

def best_f1_threshold(ytrue, prob, weights):
    p, r, thresholds = precision_recall_curve(ytrue, prob, sample_weight=weights)
    f1 = 2*p[:-1]*r[:-1]/(p[:-1]+r[:-1]+1e-12)
    if len(thresholds)==0: return .5, np.nan
    i = int(np.nanargmax(f1))
    return float(thresholds[i]), float(f1[i])

threshold_rows, oof_probs = [], {}
for name, st in studies.items():
    oof = oof_predictions(name, st.best_params)
    oof_probs[name] = oof
    threshold, score = best_f1_threshold(y_train, oof, w_train)
    threshold_rows.append({"model":name,"threshold":threshold,"training_oof_weighted_f1":score,
        "training_oof_weighted_pr_auc":average_precision_score(y_train,oof,sample_weight=w_train)})
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)
threshold_map = dict(zip(threshold_df.model, threshold_df.threshold))

## 9. Held-out test evaluation

In [ ]:
fitted, test_probs, test_preds, rows = {}, {}, {}, []
for name, st in studies.items():
    b = fit_bundle(name, st.best_params, X_train, y_train, w_train)
    fitted[name] = b
    prob = predict_bundle(b, name, X_test)
    pred = (prob >= threshold_map[name]).astype(int)
    test_probs[name], test_preds[name] = prob, pred
    rows.append({
      "model":name, "threshold":threshold_map[name], "n_test":len(y_test),
      "weighted_roc_auc":roc_auc_score(y_test,prob,sample_weight=w_test),
      "weighted_pr_auc":average_precision_score(y_test,prob,sample_weight=w_test),
      "weighted_accuracy":accuracy_score(y_test,pred,sample_weight=w_test),
      "weighted_precision":precision_score(y_test,pred,sample_weight=w_test,zero_division=0),
      "weighted_recall":recall_score(y_test,pred,sample_weight=w_test,zero_division=0),
      "weighted_f1":f1_score(y_test,pred,sample_weight=w_test,zero_division=0),
      "unweighted_accuracy":accuracy_score(y_test,pred),
      "unweighted_precision":precision_score(y_test,pred,zero_division=0),
      "unweighted_recall":recall_score(y_test,pred,zero_division=0),
      "unweighted_f1":f1_score(y_test,pred,zero_division=0)})
results_df = pd.DataFrame(rows).sort_values("weighted_pr_auc",ascending=False)
display(results_df)
for name in studies:
    print("\n",name,"threshold:",round(threshold_map[name],4))
    print(classification_report(y_test,test_preds[name],
          target_names=["Not Any Wasting","Any Wasting"],zero_division=0))
    print("Confusion matrix:\n",confusion_matrix(y_test,test_preds[name]))

## 10. ROC and precision-recall curves

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,5))
for name, prob in test_probs.items():
    RocCurveDisplay.from_predictions(y_test,prob,sample_weight=w_test,name=name,ax=ax[0])
    PrecisionRecallDisplay.from_predictions(y_test,prob,sample_weight=w_test,name=name,ax=ax[1])
ax[0].set_title("Weighted ROC — held-out test")
ax[1].set_title("Weighted precision-recall — held-out test")
plt.tight_layout()
plt.show()

## 11. Feature importance

Importance scales differ across algorithms; compare rankings within each model rather than raw magnitudes across models.

In [ ]:
importance_frames = []
for name,b in fitted.items():
    if name=="CatBoost":
        vals=b["model"].get_feature_importance()
        names=PREDICTORS
    else:
        vals=b["model"].feature_importances_
        names=b["prep"].get_feature_names_out()
    z=pd.DataFrame({"feature":names,"importance":vals})
    z["model"]=name
    importance_frames.append(z)
importance_df=pd.concat(importance_frames,ignore_index=True)
for name in studies:
    top=importance_df[importance_df.model==name].nlargest(15,"importance").sort_values("importance")
    plt.figure(figsize=(8,6))
    plt.barh(top.feature,top.importance)
    plt.title(name+" — top 15 feature importances")
    plt.tight_layout()
    plt.show()

## 12. Save outputs and run manifest

In [ ]:
OUTPUT_DIR=Path("outputs")/"any_wasting_smote_weighted_model_comparison"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
results_df.to_csv(OUTPUT_DIR/"heldout_model_comparison.csv",index=False)
threshold_df.to_csv(OUTPUT_DIR/"training_oof_thresholds.csv",index=False)
importance_df.to_csv(OUTPUT_DIR/"feature_importance.csv",index=False)
for name,st in studies.items():
    st.trials_dataframe().to_csv(OUTPUT_DIR/(name.lower()+"_optuna_trials.csv"),index=False)
    with open(OUTPUT_DIR/(name.lower()+"_best_params.json"),"w") as f: json.dump(st.best_params,f,indent=2)
manifest={
 "target":TARGET,"predictors":PREDICTORS,"weight_column":WEIGHT_COL,
 "weight_use":"Child sample weight in model fitting and weighted metrics; normalized within training fit.",
 "smote":"SMOTENC within training folds only.",
 "synthetic_weight_rule":"Mean original minority-class child weight in the corresponding training fold.",
 "models":list(studies),"optuna_trials_per_model":OPTUNA_TRIALS,"cv_folds":CV_FOLDS,
 "threshold_selection":"Maximize survey-weighted F1 on training-only OOF probabilities.",
 "thresholds":threshold_map,"heldout_metrics":results_df.to_dict(orient="records"),
 "caveat":"Synthetic records have approximate weights; SMOTE does not preserve complex survey design."
}
with open(OUTPUT_DIR/"run_manifest.json","w") as f: json.dump(manifest,f,indent=2)
print("Saved:",OUTPUT_DIR.resolve())

## 13. Methodological cautions

- SMOTENC is applied inside each training fold only; validation and test rows are never resampled.
- The child sample weight is not a class-balancing weight. It represents the survey's child sampling weight and is used in fitting and weighted evaluation.
- Synthetic observations have no design weight. Assigning the mean minority-class weight is an approximation; results require a no-SMOTE sensitivity analysis.
- SMOTE changes the sample distribution and does not preserve MICS strata, clusters or other complex survey-design features. This notebook is a predictive workflow, not a replacement for survey-design-based population inference.
- Thresholds maximize weighted F1 on training-only OOF predictions. They are data-dependent operating points, not universal public-health cut-offs.
- Report weighted and unweighted metrics; do not rely on accuracy alone.